# Avance 4 · Regresión de la potencia en la fase 2

**Bioactive AI** · Equipo 7 · Maestría en Inteligencia Artificial Aplicada (TC5035.10)

Este cuaderno es el compañero del [04_Models](04_Models.ipynb) para la fase 2. Allí la comparativa de seis modelos es de **clasificación a 10 µM**. Aquí cambiamos a una **regresión**, por una aclaración que nos hizo la Dra. Mariana el fin de semana sobre la variable objetivo y que cambia el enfoque de la fase 2: ya no se predice si un compuesto es activo o inactivo, sino **cuánta potencia tiene** (`pActividad`), y la clase se deriva de ella al final ([2.9](../docs/decisiones.md#d2-9)). Por eso la comparativa de clasificación del 04 queda como el piso.

Como ahora es una regresión, su baseline se agregó al notebook [03b](03b_Baseline_regresion_actividad.ipynb). Ahí se probó un modelo sencillo que predice la potencia con las mediciones exactas (una recta, Ridge) y otros dos que consideran las mediciones censuradas: uno que toma la cota como si fuera el valor medido (Ridge con la cota imputada) y otro que usa la verosimilitud de la censura (regresión de Tobit), junto con la media como piso.

Aquí son seis modelos individuales, sin ensambles, entrenados con las mediciones exactas.


## 1. Preparación

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"          # un solo hilo por modelo: la columna de tiempos es comparable

import platform
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.stats import spearmanr
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import BayesianRidge, Ridge
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
PROCESADOS = RAIZ / "data" / "processed"

from src.evaluation.regresion import concordancia_harrell, rmse_exactas

# Versiones con las que se ejecutó: tiene que decir Python 3.12.
print(f"sistema  {platform.platform()}")
print(f"python   {platform.python_version()}")
for m in (np, pd, sklearn, scipy):
    print(f"{m.__name__:<8} {m.__version__}")

sistema  macOS-27.0.1-arm64-arm-64bit
python   3.12.12
numpy    2.5.4
pandas   3.0.6
sklearn  1.9.1
scipy    1.18.1


### El código de las métricas

Este cuaderno usa dos métricas que no vienen en scikit-learn. Su código se muestra aquí para que se lea sin abrir otro archivo; **lo que se ejecuta es el de `src/evaluation/regresion.py`**, que importa la celda de arriba, y es el mismo que prueba `tests/test_regresion.py` y que reutilizará el Avance 5. La prueba `tests/test_04b_codigo.py` comprueba que el código de esta celda y el de `src/` sean idénticos, de modo que lo que se lee es lo que corrió.

- **Concordancia de Harrell.** Compara solo los pares de mediciones cuyos intervalos no se solapan (`superior_i < inferior_j`), porque solo de esos se sabe cuál es más potente. Un par es concordante si el modelo predice más potencia para el más potente, y un empate cuenta medio. Es la única métrica que usa también las mediciones censuradas, y por eso es la principal.
- **RMSE sobre las exactas.** El error cuadrático medio, solo con las mediciones exactas, porque un `> 10 µM` no tiene un valor contra el cual restar. Devuelve también cuántas son.

```python
def _exactas(limite_inferior: np.ndarray, limite_superior: np.ndarray) -> np.ndarray:
    inf = np.asarray(limite_inferior, dtype=float)
    sup = np.asarray(limite_superior, dtype=float)
    return np.isfinite(inf) & (inf == sup)


def concordancia_harrell(limite_inferior: np.ndarray, limite_superior: np.ndarray,
                         prediccion: np.ndarray, bloque: int = 2000) -> float:
    """Fracción de pares comparables que el modelo ordena bien.

    Un par (i, j) es **comparable** si los intervalos no se solapan
    (`superior_i < inferior_j`): solo entonces se sabe que j es más potente. Es
    concordante si la predicción de j es mayor; un empate de predicciones cuenta
    medio. Con todas las mediciones exactas es el índice de concordancia
    ordinario. Devuelve `nan` si no hay pares comparables.
    """
    inf = np.asarray(limite_inferior, dtype=float)
    sup = np.asarray(limite_superior, dtype=float)
    pred = np.asarray(prediccion, dtype=float)
    concordantes = 0.0
    comparables = 0
    for a in range(0, len(pred), bloque):
        s = sup[a:a + bloque, None]
        p = pred[a:a + bloque, None]
        comparable = s < inf[None, :]          # i (fila) es menos potente que j
        comparables += int(comparable.sum())
        concordantes += float((comparable & (p < pred[None, :])).sum())
        concordantes += 0.5 * float((comparable & (p == pred[None, :])).sum())
    return concordantes / comparables if comparables else float("nan")


def rmse_exactas(limite_inferior: np.ndarray, limite_superior: np.ndarray,
                 prediccion: np.ndarray) -> tuple[float, int]:
    """RMSE sobre las mediciones exactas; devuelve `(rmse, n)`. Es solo del
    subconjunto no censurado, y así hay que declararlo donde se reporte."""
    e = _exactas(limite_inferior, limite_superior)
    if not e.any():
        return float("nan"), 0
    error = (np.asarray(limite_inferior, dtype=float)[e]
             - np.asarray(prediccion, dtype=float)[e])
    return float(np.sqrt(np.mean(error ** 2))), int(e.sum())
```


### Los datos

La unidad es el par compuesto-diana. Se usan los mismos pares de entrenamiento y validación que el 04 (la prueba reservada no se toca) y la diana entra como columnas de 0 y 1. Las filas son las del baseline del 03b: **ninguna se descarta por no decidir a 10 µM**, y salen solo las que no informan nada (sin ninguna cota, o con cotas que se contradicen). Los modelos se entrenan con las exactas; la validación incluye las censuradas, porque la concordancia las usa.

In [2]:
pares = pd.read_parquet(PROCESADOS / "fase2_pares.parquet")
desc = pd.read_parquet(PROCESADOS / "fase2_descriptores_limpios.parquet")
d = pares[pares["particion"].ne("prueba")].merge(desc, on="smiles").reset_index(drop=True)
cols = [c for c in desc.columns if c != "smiles"]

inferior = d["limite_inferior"].to_numpy(float)
superior = d["limite_superior"].to_numpy(float)
util = ~(np.isneginf(inferior) & np.isposinf(superior)) & (inferior <= superior)
print(f"{len(d):,} pares de entrenamiento y validación · {(~util).sum():,} sin información y salen")

d = d[util].reset_index(drop=True)
inferior, superior = inferior[util], superior[util]
# La diana entra como columnas 0/1: sin ella el modelo no sabe contra qué se midió.
X = np.hstack([d[cols].to_numpy("float32"), pd.get_dummies(d["target_gene"]).to_numpy("float32")])
exacta = inferior == superior
y_cls = d["activo_10um"].to_numpy(float)       # la clase derivada a 10 µM; NaN si no decide
g = d["grupo_cv"].to_numpy()
procedencia = d["estrato"].to_numpy()          # solo_chembl, solo_npass, mixto
print(f"{len(d):,} pares · {exacta.sum():,} con potencia exacta ({exacta.mean():.1%}) · "
      f"{(~exacta).sum():,} solo con cotas")

56,411 pares de entrenamiento y validación · 170 sin información y salen
56,241 pares · 46,100 con potencia exacta (82.0%) · 10,141 solo con cotas


## 2. Seis modelos de regresión

| Clasificación (notebook 04) | Regresión (aquí) |
|:--|:--|
| logística | `Ridge` (regresión lineal) |
| árbol | `DecisionTreeRegressor` |
| kNN | `KNeighborsRegressor` |
| Naive Bayes | `BayesianRidge` (regresión lineal bayesiana) |
| SVM-RBF | `SVR` |
| MLP | `MLPRegressor` |

Naive Bayes no tiene versión de regresión; `BayesianRidge` es la familia bayesiana que sí la tiene. `media` (predecir siempre el promedio de las exactas) va aparte como **referencia**: es el piso y no cuenta entre los seis, porque no es un algoritmo.

**Métrica principal, fijada antes de ver los resultados: la concordancia de Harrell.** Es la única que usa también las mediciones censuradas, y la tabla se ordena por ella. Las otras son el Spearman y el RMSE sobre las exactas (declarado así: no dicen nada de las censuradas), y el MCC y el AUROC de la clase derivada, cortando la potencia predicha en 5 (= 10 µM), para poder compararlos con el 04.

**El mínimo que hay que superar** viene del [03b](03b_Baseline_regresion_actividad.ipynb): el mejor de cada métrica entre sus baselines, concordancia 0.700, Spearman 0.531 y RMSE 1.156. Un modelo que no los supere no aporta sobre una recta.


### Qué hace cada modelo por dentro

Todos reciben los 201 descriptores del compuesto y la diana como columnas de 0 y 1, y aprenden a predecir la `pActividad` con las mediciones exactas. Los que dependen de distancias o de magnitudes (Ridge, `BayesianRidge`, kNN, SVR y MLP) llevan las entradas escaladas dentro de la canalización, de modo que cada pliegue aprende su propio escalado.

| Modelo | Qué hace por dentro | Para qué sirve aquí |
|:--|:--|:--|
| **`media`** (referencia) | No aprende: devuelve siempre el promedio de las exactas del entrenamiento | El piso. Un modelo que no le gane no aprendió nada |
| **`Ridge`** | Una recta: la potencia es una suma ponderada de las entradas. Elige los pesos que minimizan el error cuadrático, con una penalización que los encoge para que ninguno domine | La regresión lineal que cualquiera haría; sus pesos se pueden leer |
| **`BayesianRidge`** | La misma recta, pero trata los pesos como cantidades inciertas con una distribución a priori y estima sola cuánta penalización conviene y cuánto ruido tienen los datos | Una recta sin penalización que fijar a mano. Es la familia bayesiana que sí tiene versión de regresión |
| **`DecisionTreeRegressor`** | Hace preguntas de sí o no sobre una entrada (por ejemplo, «¿TPSA menor que 67?») y divide los datos hasta que cada hoja tiene al menos 5 casos; predice el promedio de la hoja | Capta reglas y cruces entre entradas; se lee de arriba abajo. Se pasa de ajustado si crece mucho |
| **`KNeighborsRegressor`** (15 vecinos) | No ajusta nada: para un par nuevo busca los 15 más parecidos del entrenamiento (distancia euclidiana) y promedia su potencia | Parte de que moléculas parecidas contra la misma diana tienen potencia parecida |
| **`SVR`** (núcleo RBF) | Ajusta una función suave que tolera errores pequeños dentro de un margen y solo penaliza los que lo exceden; el núcleo RBF mide la semejanza entre dos moléculas de forma no lineal | Capta relaciones curvas sin construirlas a mano. Es el más lento: su costo crece entre cuadrático y cúbico con las filas |
| **`MLPRegressor`** (una capa de 64 neuronas) | Una red pequeña: combina las entradas en 64 unidades, aplica una no linealidad a cada una y las vuelve a combinar; ajusta los pesos por descenso de gradiente sobre el error cuadrático | Una función flexible que aprende cruces de entradas por su cuenta. Es la menos legible |

Ninguno es un ensamble: cada uno es un solo modelo. La recta con censura (Tobit) no está aquí, sino en el [03b](03b_Baseline_regresion_actividad.ipynb), donde se explica cómo usa las mediciones censuradas.


In [3]:
MODELOS = {
    "ridge":          make_pipeline(MinMaxScaler(), Ridge()),
    "arbol":          DecisionTreeRegressor(min_samples_leaf=5),
    "knn":            make_pipeline(MinMaxScaler(), KNeighborsRegressor(n_neighbors=15)),
    "bayesian_ridge": make_pipeline(StandardScaler(), BayesianRidge()),
    "svr":            make_pipeline(MinMaxScaler(), SVR()),
    "mlp":            make_pipeline(MinMaxScaler(), MLPRegressor(hidden_layer_sizes=(64,), max_iter=600)),
}
REFERENCIA = {"media": DummyRegressor()}        # el piso: no aprende nada
SEMILLAS = (0, 1, 2)
CORTE = 5.0                                      # pActividad 5 = 10 µM, el mismo corte del 04
MINIMOS = {"concordancia": 0.700, "spearman": 0.531, "rmse": 1.156}     # del 03b


def medir(inf, sup, clase, pred):
    """Concordancia con todas las mediciones; Spearman y RMSE con las exactas; MCC y AUROC de la clase derivada."""
    exa = inf == sup
    pe = pred[exa]
    rho = np.nan if np.ptp(pe) == 0 else spearmanr(inf[exa], pe).statistic   # nan si predice una constante
    rmse, _ = rmse_exactas(inf, sup, pred)
    fila = {"concordancia": concordancia_harrell(inf, sup, pred), "spearman": rho, "rmse": rmse,
            "mcc": np.nan, "auroc": np.nan}
    decide = ~np.isnan(clase)
    if len(np.unique(clase[decide])) == 2:
        fila["mcc"] = matthews_corrcoef(clase[decide], pred[decide] >= CORTE)
        fila["auroc"] = roc_auc_score(clase[decide], pred[decide])
    return fila


filas = []
estratificar = np.where(np.isnan(y_cls), -1, y_cls).astype(int)
for semilla in SEMILLAS:
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=semilla)
    for k, (ajuste, val) in enumerate(cv.split(X, estratificar, g), 1):
        ajuste_ex = ajuste[exacta[ajuste]]                       # se entrena solo con las exactas
        for nombre, modelo in {**MODELOS, **REFERENCIA}.items():
            m = clone(modelo)
            # La semilla fija también el azar del modelo (árbol, MLP), no solo el reparto.
            m.set_params(**{p: semilla for p in m.get_params() if p.endswith("random_state")})
            reloj = time.perf_counter()
            m.fit(X[ajuste_ex], inferior[ajuste_ex])
            seg = time.perf_counter() - reloj
            pred = m.predict(X[val])
            base = {"modelo": nombre, "semilla": semilla, "pliegue": k, "seg": seg}
            filas.append({**base, "estrato": "todos",
                          **medir(inferior[val], superior[val], y_cls[val], pred)})
            for e in np.unique(procedencia[val]):
                sel = procedencia[val] == e
                ys = y_cls[val][sel]
                ys = ys[~np.isnan(ys)]
                if min(ys.sum(), (1 - ys).sum()) >= 20:         # estrato con 20 casos de cada clase
                    filas.append({**base, "estrato": e,
                                  **medir(inferior[val][sel], superior[val][sel], y_cls[val][sel], pred[sel])})
res = pd.DataFrame(filas)
print(f"{len(res):,} filas: {res['modelo'].nunique()} modelos × {len(SEMILLAS)} semillas × 5 pliegues, por estrato")

420 filas: 7 modelos × 3 semillas × 5 pliegues, por estrato


## 3. Resultados

In [4]:
def tabla(res, estrato="todos"):
    """Media por modelo, ordenada por la métrica principal. `desv_concordancia` es dispersión entre
    evaluaciones, no error estándar: las semillas comparten filas."""
    r = res[res["estrato"] == estrato]
    t = r.groupby("modelo")[["concordancia", "spearman", "rmse", "mcc", "auroc", "seg"]].mean()
    t.insert(1, "desv_concordancia", r.groupby("modelo")["concordancia"].std())
    t["supera_minimos"] = ((t["concordancia"] > MINIMOS["concordancia"]).astype(int)
                           + (t["spearman"] > MINIMOS["spearman"]).astype(int)
                           + (t["rmse"] < MINIMOS["rmse"]).astype(int))
    return t.sort_values("concordancia", ascending=False).round(3)


tabla_todos = tabla(res)
tabla_todos

,concordancia,desv_concordancia,spearman,rmse,mcc,auroc,seg,supera_minimos
modelo,,,,,,,,
knn,0.739,0.004,0.660,1.010,0.341,0.805,0.012,3
svr,0.738,0.004,0.640,1.034,0.353,0.819,50.741,3
mlp,0.735,0.006,0.640,1.092,0.370,0.800,16.647,3
bayesian_ridge,0.692,0.006,0.530,1.157,0.225,0.776,0.119,0
ridge,0.691,0.006,0.528,1.148,0.222,0.774,0.022,1
arbol,0.670,0.007,0.484,1.308,0.286,0.717,2.047,0
media,0.500,0.000,NaN,1.337,0.000,0.500,0.004,0


### Por procedencia

Un promedio puede esconder cómo le va a los naturales. En el 03b la concordancia de las rectas sube de 0.53 a 0.64 solo en los pares que solo tiene NPASS, y el control del Avance 3 mostró que el modelo separa mucho peor esos pares que los de ChEMBL.

In [5]:
por_procedencia = {e: tabla(res, e) for e in ("solo_chembl", "mixto", "solo_npass")
                   if (res["estrato"] == e).any()}
pd.concat(por_procedencia, names=["procedencia"])[["concordancia", "desv_concordancia", "spearman", "rmse", "mcc", "auroc"]]

concordancia  desv_concordancia  spearman   rmse  \
procedencia modelo                                                             
solo_chembl knn                    0.733              0.004     0.644  1.013   
            svr                    0.730              0.005     0.620  1.040   
            mlp                    0.729              0.006     0.627  1.091   
            bayesian_ridge         0.681              0.007     0.502  1.148   
            ridge                  0.680              0.007     0.501  1.151   
            arbol                  0.665              0.007     0.468  1.312   
            media                  0.500              0.000       NaN  1.325   
mixto       mlp                    0.570              0.024     0.203  1.145   
            knn                    0.565              0.025     0.294  0.981   
            svr                    0.564              0.022     0.211  0.988   
            arbol                  0.532              0.016     0.104  1.266   
            ridge                  0.515              0.030     0.052  1.091   
            bayesian_ridge         0.510              0.031     0.044  1.092   
            media                  0.500              0.000       NaN  1.498   
solo_npass  mlp                    0.551              0.031     0.249  1.065   
            svr                    0.535              0.028     0.313  0.813   
            bayesian_ridge         0.532              0.029     0.240  1.292   
            ridge                  0.522              0.028     0.236  1.056   
            media                  0.500              0.000       NaN  1.640   
            arbol                  0.498              0.037     0.194  1.175   
            knn                    0.464              0.019     0.342  0.859   

                              mcc  auroc  
procedencia modelo                        
solo_chembl knn             0.320  0.798  
            svr             0.318  0.808  
            mlp             0.345  0.793  
            bayesian_ridge  0.172  0.762  
            ridge           0.169  0.761  
            arbol           0.269  0.711  
            media           0.000  0.500  
mixto       mlp             0.142  0.598  
            knn             0.138  0.615  
            svr             0.128  0.601  
            arbol           0.072  0.537  
            ridge          -0.006  0.507  
            bayesian_ridge -0.003  0.501  
            media           0.000  0.500  
solo_npass  mlp             0.058  0.571  
            svr             0.015  0.542  
            bayesian_ridge  0.013  0.528  
            ridge           0.014  0.517  
            media           0.000  0.500  
            arbol           0.003  0.512  
            knn            -0.013  0.490

## 3.1 Lectura

La corrida la ejecutó Artemio Padilla en macOS (arm64) con Python 3.12.12 y las versiones del lock; la primera celda lo imprime. No se tocó el código del cuaderno.

**Lo que vemos.**

1. **Tres modelos se despegan del resto, y entre ellos no hay ganador.** El kNN, el SVR y el MLP superan los tres mínimos que dejó el baseline del 03b (concordancia 0.739, 0.738 y 0.735; Spearman 0.660, 0.640 y 0.640; RMSE 1.010, 1.034 y 1.092). La diferencia entre ellos es de apenas 0.001 a 0.004, menos de lo que se mueve el resultado de una evaluación a otra (0.004 a 0.006), así que por la métrica principal están empatados. Las dos rectas se quedan donde ya estaba la del 03b (0.691 y 0.692) y el árbol en 0.670.
2. **Los modelos más flexibles ayudan con ChEMBL, pero no con los productos naturales.** En los pares de solo ChEMBL esos tres suben a 0.729–0.733, contra 0.680 de las rectas. En los pares que solo tiene NPASS, que son los naturales que nos interesan, los seis modelos están cerca del azar: entre 0.464 y 0.551 de concordancia (el kNN incluso queda por debajo de 0.500).
3. **Parece un problema de con qué se entrena, más que del modelo.** El baseline con censura del 03b, que es solo una recta, saca 0.639 en esos mismos pares; el mejor de estos seis, el MLP, saca 0.551. Es una comparación indicativa y no de la misma corrida: el 03b usó cinco semillas y Linux, y esta tabla tres semillas y macOS.
4. **Acertar en las exactas no alcanza.** En los pares de NPASS el kNN y el SVR son los que menos se equivocan en las mediciones exactas (RMSE de 0.859 y 0.813) y los que mejor las ordenan (Spearman de 0.342 y 0.313), pero tienen la concordancia más baja. Nuestra lectura, que este cuaderno no demuestra, es que predicen bien lo exacto y colocan mal lo censurado, que es la mayor parte de los negativos de NPASS.
5. **Ningún modelo se acerca al techo de ruido, y eso es buena señal.** Repetir el mismo ensayo da diferencias de unas 0.7 unidades logarítmicas: ese es el techo que nos recomendó la Dra. Mariana, y es lo mejor que se le puede pedir a un modelo. Nuestros errores van de 1.010 a 1.308, bastante por encima, así que no hay indicio de que algo se esté filtrando.
6. **Los tiempos.** Por ajuste, con un solo hilo y nada más corriendo: el kNN tarda 0.012 s, `ridge` 0.022 s, `bayesian_ridge` 0.12 s, el árbol 2.0 s, el MLP 16.6 s y el SVR 50.7 s. Son de un procesador de Apple, así que no son comparables con los del 04, que se midieron en Docker.

**Lo que sigue.** Vamos a proponer otro modelo que sí trata las mediciones censuradas, pero es un ensamble; como este avance pide modelos individuales, lo agregaremos después, en el Avance 5. También quedan para el Avance 5 las sensibilidades que declaramos de antemano (sin las relaciones inferidas, solo dirección consistente, con y sin DrugMatrix) y el corte de 1 µM junto al de 10 µM. Lo de DrugMatrix, en particular, todavía no se ha medido.


In [6]:
# Registro en experimentos/ (decisión 2.9: la potencia como variable principal).
from src.tracking import Corrida

with Corrida("comparativa-regresion-seis-modelos", decision="2.9",
             params={"modelos": list(MODELOS), "referencia": list(REFERENCIA), "pliegues": 5,
                     "entrena_con": "solo exactas", "objetivo": "pActividad",
                     "metrica_principal": "concordancia", "corte_mcc": CORTE,
                     "minimos_03b": MINIMOS, "estratos": "procedencia"},
             semillas=list(SEMILLAS),
             datos=[PROCESADOS / "fase2_pares.parquet",
                    PROCESADOS / "fase2_descriptores_limpios.parquet"]) as c:
    c.metricas({"todos": tabla_todos.to_dict(orient="index"),
                **{e: t.to_dict(orient="index") for e, t in por_procedencia.items()}})

print(c.carpeta.name)

2026-10-25-artemio-2.9-comparativa-regresion-seis-modelos


# Declaración de uso de IA y contribución

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic. (2026). Claude Code [Agente de programación]. https://claude.com/claude-code | Nos apoyó con la generación de código, con la investigación de modelos que consideran la censura y con el criterio de las filas que entran al análisis | Las secciones que nos ayudó a implementar fueron celdas de código | La selección de los modelos, la revisión de los resultados, la interpretación de las métricas y los comentarios generales fueron del equipo |

### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Ingrid Pamela Ruiz Puga | Implementó un modelo (`BayesianRidge`), conectó el baseline del 03b y propuso implementar las métricas de evaluación que sugirió la Dra. Mariana |
| Artemio Santiago Padilla Robles | Revisó y ejecutó el notebook en el entorno del lock |
| José Antonio Vázquez Martínez | Implementó cinco de los seis modelos del cuaderno |

### Decisiones principales

**Decisión 1. El sexto algoritmo es `BayesianRidge`, y `media` no cuenta como modelo.**

- Justificación. La rúbrica pide seis algoritmos distintos y `media` predice siempre el promedio, así que es el piso y no un algoritmo. `BayesianRidge` ocupa en regresión el casillero que Naive Bayes ocupa en la tabla de clasificación del cuaderno 04, con lo que las dos tablas se corresponden.
- Alternativa descartada y motivo. `ElasticNet` y `PLSRegression`, que también cabían pero son lineales como la `Ridge` que ya está en la tabla, así que habrían repetido familia en vez de cubrir una nueva.

**Decisión 2. La métrica principal es la concordancia de Harrell, fijada antes de ver los resultados.**

- Justificación. De las tres métricas acordadas para la regresión es la única que tolera las mediciones censuradas, así que describe todos los pares y no solo la mitad exacta. Es además la vara con la que se compara contra el baseline con censura del 03b.
- Alternativa descartada y motivo. Ordenar por Spearman, que es lo que hacía la primera versión del cuaderno, y que solo mira las mediciones exactas.

**Decisión 3. Este cuaderno entrena solo con las mediciones exactas y se reporta como referencia sin censura.**

- Justificación. Una regresión sencilla no sabe qué hacer con un «> 10 µM» y escribirle un número sería inventar un dato. Deja fuera 7,535 de los 15,554 pares inactivos, el 48.4%, que existen solo como cotas y son justamente los compuestos flojos.
- Alternativa descartada y motivo. Usar el modelo de Tobit, que sí aprovecha las cotas. Vive en el 03b como baseline con censura, y la comparación entre los dos cuadernos es la que aísla cuánto cuesta descartar los censurados.
